In [ ]:
!pip install -qqq 'pip' 'crewai[tools]==0.28.8' 'duckduckgo-search==5.3.0' 'langchain-groq==0.1.3' --progress-bar off


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os
import re
from datetime import datetime
import pandas as pd
import requests
from crewai import Agent, Crew, Process, Task
from crewai_tools import tool
from google.colab import userdata
from langchain.agents import load_tools
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_groq import ChatGroq

def format_response(response: str) -> str:
  entries=re.split(r"(?<=]), (?=\[)", response)
  return [entry.strip("[]") for entry in entries]

os.environ ["GROQ_API_KEY"] = userdata.get("lama3")

In [ ]:
search_tool = DuckDuckGoSearchResults(backend="news",num_results=10)

In [ ]:
response = search_tool.run("Bitcoin")
format_response(response)

['snippet: US exchange-traded funds investing directly in Bitcoin attracted net inflows for an unprecedented 18th straight day, a spurt of demand that has helped to lift the largest digital asset toward a record high., title: US Bitcoin ETFs Post Longest Run of Inflows as Token Nears Record, link: https://www.msn.com/en-us/money/other/us-bitcoin-etfs-post-longest-run-of-inflows-as-token-nears-record/ar-BB1nMTes, date: 2024-06-07T05:02:24+00:00, source: Bloomberg L.P. on MSN.com',
 'snippet: Healthcare manufacturer Semler Scientific now owns 828 Bitcoin, worth $58.5 million after its latest purchase, and says it could offer $150 million in debt securities to buy even more., title: Semler Scientific now holds 828 Bitcoin and has $150M plan to buy more, link: https://cointelegraph.com/news/semler-scientific-now-holds-828-bitcoin-150m-plan-buy-more, date: 2024-06-07T04:27:00+00:00, source: CoinTelegraph',
 "snippet: Bitcoin's unparalleled security and resilience make it a prime candidate f

In [ ]:
human_tools = load_tools(["human"])

In [ ]:
import requests
import pandas as pd

def get_daily_closing_prices(ticker: str) -> pd.DataFrame:
    api_key = 'NS3TZ8SYWMMHH6SJ'
    url = f'https://www.alphavantage.co/query?function=DIGITAL_CURRENCY_DAILY&symbol={ticker}&market=USD&apikey={api_key}'
    response = requests.get(url)

    if response.status_code != 200:
        raise Exception("Failed to fetch data from Alpha Vantage API")

    data = response.json()

    if "Time Series (Digital Currency Daily)" not in data:
        raise Exception("Invalid response from Alpha Vantage API")

    price_data = data["Time Series (Digital Currency Daily)"]

    daily_close_prices = {
        date: prices["4. close"] for (date, prices) in price_data.items()
    }

    df = pd.DataFrame.from_dict(daily_close_prices, orient="index", columns=["price"])
    df.index = pd.to_datetime(df.index)
    df["price"] = pd.to_numeric(df["price"])

    return df


In [ ]:
price_df = get_daily_closing_prices("BTC")

price_df.head()

,price
2024-06-07,70851.19
2024-06-06,70773.64
2024-06-05,71121.11
2024-06-04,70542.32
2024-06-03,68791.00


In [ ]:
text_output = []

for date, row in price_df.head(10).iterrows():
  text_output.append(f"{date.strftime('%Y-%m-%d')} - {row['price']:.2f}")

formatted_text = "\n".join(text_output)

print(formatted_text)

2024-06-07 - 70851.19
2024-06-06 - 70773.64
2024-06-05 - 71121.11
2024-06-04 - 70542.32
2024-06-03 - 68791.00
2024-06-02 - 67735.52
2024-06-01 - 67719.29
2024-05-31 - 67472.41
2024-05-30 - 68338.58
2024-05-29 - 67569.45


In [ ]:
@tool("price tool")

def cryptocurrency_price_tool(ticker_symbol: str) -> str:

  """Get daily closing price for a given cryptocurrency ticker symbol for the previous 60 days"""

  price_df = get_daily_closing_prices(ticker_symbol)

  text_output = []

  for date, row in price_df.head(60).iterrows():
    text_output.append(f"{date.strftime('%Y-%m-%d')} - {row['price']:.2f}")


  return "\n".join(text_output)

In [ ]:
@tool("price tool")

def cryptocurrency_news_tool(ticker_symbol: str) -> str:

  """Get news for a given   cryptocurrency ticker symbol"""
  return search_tool.run(ticker_symbol + " cryptocurrency")

In [ ]:
llm = ChatGroq(temperature=0,model_name="llama3-70b-8192")

In [ ]:
%%time
system = "You are experienced Machine Learning & AI Engineer."
human = "{text}"
prompt = ChatPromptTemplate.from_messages([("system", system), ("human", human)])

chain = prompt | llm
response = chain.invoke({"text": "How to increase inference speed for a 7B LLM?"})


CPU times: user 71.6 ms, sys: 1.65 ms, total: 73.2 ms
Wall time: 2.07 s


In [ ]:
print(response.content)

As a seasoned Machine Learning & AI Engineer, I'd be happy to help you optimize the inference speed for a 7B Large Language Model (LLM).

Here are some strategies to increase inference speed for a 7B LLM:

1. **Model Pruning**: Remove redundant or unnecessary weights and connections in the model to reduce its size and computational requirements. This can be done using techniques like magnitude-based pruning, L1 regularization, or iterative pruning.
2. **Knowledge Distillation**: Train a smaller, simpler model (the student) to mimic the behavior of the large 7B LLM (the teacher). This can reduce the model size and inference time while preserving the original model's performance.
3. **Quantization**: Represent the model's weights and activations using fewer bits (e.g., int8 instead of float32). This reduces memory usage and can lead to faster inference times. Techniques like post-training quantization or quantization-aware training can be used.
4. **Tensor Train Decomposition**: Decompos

In [ ]:
customer_communicator = Agent(
    role="Senior cryptocurrency customer communicator",
    goal="Find which cryptocurrency the customer is interested in",
    backstory="""You're highly experienced in communicating about cryptocurrencies
    and blockchain technology with customers and their research needs""",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=5,
    memory=True,
    tools=human_tools,
)

In [ ]:

news_analyst = Agent(
    role="Cryptocurrency News Analyst",
    goal="""Get news for a given cryptocurrency. Write 1 paragraph analysis of
    the market and make prediction - up, down or neutral.""",
    backstory="""You're an expert analyst of trends based on cryptocurrency news.
    You have a complete understanding of macroeconomic factors, but you specialize
    into analyzing news.
    """,
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=5,
    memory=True,
    tools=[cryptocurrency_news_tool],
)

In [ ]:
price_analyst = Agent(
    role="Cryptocurrency Price Analyst",
    goal="""Get historical prices for a given cryptocurrency. Write 1 paragraph analysis of
    the market and make prediction - up, down or neutral.""",
    backstory="""You're an expert analyst of trends based on cryptocurrency
    historical prices. You have a complete understanding of macroeconomic factors,
    but you specialize into technical analys based on historical prices.
    """,
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=5,
    memory=True,
    tools=[cryptocurrency_price_tool],
)



In [ ]:
writer = Agent(
    role="Cryptocurrency Report Writer",
    goal="""Write 1 paragraph report of the cryptocurrency market.""",
    backstory="""
    You're widely accepted as the best cryptocurrency analyst that
    understands the market and have tracked every asset for more than 10 years. Your trends
    analysis are always extremely accurate.

    You're also master level analyst in the traditional markets and have deep understanding
    of human psychology. You understand macro factors and combine those multiple
    theories - e.g. cycle theory. You're able to hold multiple opininons when analysing anything.

    You understand news and historical prices, but you look at those with a
    healthy dose of skepticism. You also consider the source of news articles.

    Your most well developed talent is providing clear and concise summarization
    that explains very complex market topics in simple to understand terms.

    Some of your writing techniques include:

    - Creating a bullet list (executive summary) of the most importannt points
    - Distill complex analyses to their most important parts

    You writing transforms even dry and most technical texts into
    a pleasant and interesting read.""",
    llm=llm,
    verbose=True,
    max_iter=5,
    memory=True,
    allow_delegation=False,
)

In [ ]:
get_cryptocurrency = Task(
    description=f"Ask which cryptocurrency the customer is interested in.",
    expected_output="""Cryptocurrency symbol that the human wants you to research e.g. BTC.""",
    agent=customer_communicator,
)


In [ ]:
get_news_analysis = Task(
    description=f"""
    Use the search tool to get news for the cryptocurrency

    The current date is {datetime.now()}.

    Compose the results into a helpful report""",
    expected_output="""Create 1 paragraph report for the cryptocurrency,
    along with a prediction for the future trend
    """,
    agent=news_analyst,
    context=[get_cryptocurrency],
)


In [ ]:
get_price_analysis = Task(
    description=f"""
    Use the price tool to get historical prices

    The current date is {datetime.now()}.

    Compose the results into a helpful report""",
    expected_output="""Create 1 paragraph summary for the cryptocurrency,
    along with a prediction for the future trend
    """,
    agent=price_analyst,
    context=[get_cryptocurrency],
)

In [ ]:
write_report = Task(
    description=f"""Use the reports from the news analyst and the price analyst to
    create a report that summarizes the cryptocurrency""",
    expected_output="""1 paragraph report that summarizes the market and
    predicts the future prices (trend) for the cryptocurrency""",
    agent=writer,
    context=[get_news_analysis, get_price_analysis],
)


In [ ]:


crew = Crew(
    agents=[customer_communicator, price_analyst, news_analyst, writer],
    tasks=[get_cryptocurrency, get_news_analysis, get_price_analysis, write_report],
    verbose=2,
    process=Process.sequential,
    full_output=True,
    share_crew=False,
    manager_llm=llm,
    max_iter=15,
)


results = crew.kickoff()



 [DEBUG]: == Working Agent: Senior cryptocurrency customer communicator
 [INFO]: == Starting Task: Ask which cryptocurrency the customer is interested in.


> Entering new CrewAgentExecutor chain...
Thought: I need to ask the customer which cryptocurrency they are interested in.

Action: human
Action Input: {"question": "Hello! I'm excited to help you with your cryptocurrency research. Which cryptocurrency are you interested in?"}

Hello! I'm excited to help you with your cryptocurrency research. Which cryptocurrency are you interested in?
bitcoin
 

bitcoin

Thought: I think I have the answer, but I want to confirm.

Action: human
Action Input: {"question": "Just to confirm, you're interested in Bitcoin, correct?"}

Just to confirm, you're interested in Bitcoin, correct?
yes
 

yes

Thought: I now know the final answer
Final Answer: BTC

> Finished chain.
 [DEBUG]: == [Senior cryptocurrency customer communicator] Task output: BTC


 [DEBUG]: == Working Agent: Cryptocurrency News Analy